# 0a. Kernels, Mean Shift, and Vector Fields

Drifting Models are built out of three ingredients that have nothing to do with neural networks:

1. a **kernel** $k(\mathbf{x},\mathbf{y})$ that says how similar two samples are,
2. a **mean-shift vector** that turns a cloud of samples into a vector field,
3. the idea of **flowing points along a vector field**.

If you understand these three things, the drifting field
$$\mathbf{V}_{p,q}(\mathbf{x}) = \mathbf{V}^{+}_{p}(\mathbf{x}) - \mathbf{V}^{-}_{q}(\mathbf{x})$$
will look completely obvious when we meet it in Notebook 1. This notebook builds them from scratch.

Reference for the whole series: [_Generative Modeling via Drifting_](https://arxiv.org/abs/2602.04770) (Deng, Li, Li, Du, He, 2026).

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110

## 1. A kernel is a similarity function

A kernel takes two points and returns a number that is large when they are close and small when they are far apart.
Two kernels matter for drifting:

**Gaussian (RBF) kernel** — squared distance in the exponent:
$$k_{\text{gauss}}(\mathbf{x},\mathbf{y}) = \exp\!\left(-\frac{\|\mathbf{x}-\mathbf{y}\|^2}{2\tau^2}\right)$$

**Laplace kernel** — plain distance in the exponent:
$$k_{\text{lap}}(\mathbf{x},\mathbf{y}) = \exp\!\left(-\frac{\|\mathbf{x}-\mathbf{y}\|}{\tau}\right)$$

The Drifting Models paper uses the **Laplace** kernel. That choice is not cosmetic — we will see in section 6 (and again in Notebook 2 and `THEORY.md`) that it changes whether the resulting field is the gradient of something.

The parameter $\tau$ is the **temperature** (or bandwidth). It sets the length scale of "nearby".

In [ ]:
def k_gauss(x, y, tau):
    return torch.exp(-((x - y) ** 2).sum(-1) / (2 * tau ** 2))

def k_lap(x, y, tau):
    return torch.exp(-((x - y) ** 2).sum(-1).sqrt() / tau)

r = torch.linspace(0, 3, 400)[:, None]        # distance along one axis
zero = torch.zeros_like(r)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for tau in [0.2, 0.5, 1.0]:
    ax[0].plot(r, k_gauss(r, zero, tau), label=f"tau={tau}")
    ax[1].plot(r, k_lap(r, zero, tau), label=f"tau={tau}")
ax[0].set_title("Gaussian kernel  exp(-d^2 / 2 tau^2)")
ax[1].set_title("Laplace kernel  exp(-d / tau)")
for a in ax:
    a.set_xlabel("distance d"); a.set_ylabel("k"); a.legend(); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Note the difference at the origin. The Gaussian is flat at $d=0$ (its derivative vanishes), the Laplace kernel has a
**cusp** — it decays at full rate immediately. The Laplace kernel therefore discriminates much more aggressively
between "very close" and "close", which is exactly what you want when the thing you are trying to detect is
*whether the generated sample sits on top of a real sample or merely near it*.

## 2. Kernel density estimation

Given samples $\{\mathbf{y}_j\}_{j=1}^{M}$ from an unknown density, the **kernel density estimate** at a query point is

$$\rho_h(\mathbf{x}) \;=\; \frac{1}{M}\sum_{j=1}^{M} K_h(\mathbf{x}-\mathbf{y}_j)$$

This is the only "density model" drifting ever uses, and it is never fitted — it is defined implicitly by the batch.

In [ ]:
def make_bimodal(n, sep=2.0, sd=0.45):
    idx = torch.randint(0, 2, (n,))
    centres = torch.tensor([[-sep, 0.0], [sep, 0.0]])
    return centres[idx] + sd * torch.randn(n, 2)

data = make_bimodal(400)

def kde(grid, samples, tau, kernel=k_gauss):
    # grid: [G, 2], samples: [M, 2]
    return kernel(grid[:, None, :], samples[None, :, :], tau).mean(-1)

g = torch.linspace(-4.5, 4.5, 160)
GX, GY = torch.meshgrid(g, g, indexing="xy")
grid = torch.stack([GX.reshape(-1), GY.reshape(-1)], -1)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
for a, tau in zip(ax, [0.15, 0.5, 1.5]):
    Z = kde(grid, data, tau).reshape(GX.shape)
    a.contourf(GX, GY, Z, levels=30, cmap="magma")
    a.scatter(data[:, 0], data[:, 1], s=3, c="cyan", alpha=0.35)
    a.set_title(f"KDE, tau={tau}"); a.set_aspect("equal")
plt.tight_layout(); plt.show()

Small $\tau$ → spiky, sees every individual sample. Large $\tau$ → smooth, blurs the two modes into one.
**This tradeoff is the single most important hyperparameter in a drifting model.** We will handle it in Notebook 3
by using several $\tau$ at once, which is what the paper does.

## 3. The mean-shift vector

Here is the central object. Given a query point $\mathbf{x}$ and samples $\{\mathbf{y}_j\}$, form the
**kernel-weighted average displacement**:

$$\mathbf{V}^{+}(\mathbf{x}) \;=\; \frac{\sum_j k(\mathbf{x},\mathbf{y}_j)\,(\mathbf{y}_j-\mathbf{x})}{\sum_j k(\mathbf{x},\mathbf{y}_j)}
\;=\; \sum_j \bar{k}(\mathbf{x},\mathbf{y}_j)\,(\mathbf{y}_j-\mathbf{x})$$

where $\bar{k}$ is the kernel normalised to sum to one over $j$. In words: *look around you, weight your
neighbours by how close they are, and step toward their weighted centre of mass.*

This is exactly Fukunaga–Hostetler / Cheng **mean shift**, the classical mode-seeking algorithm from 1975/1995.
Two implementation facts:

* $\bar{k}$ **is a softmax.** For the Laplace kernel, $\bar{k}(\mathbf{x},\mathbf{y}_j) = \mathrm{softmax}_j(-\|\mathbf{x}-\mathbf{y}_j\|/\tau)$.
  The normalising constant $Z(\mathbf{x}) = \sum_j k$ is absorbed for free. Every drifting implementation is a softmax over a distance matrix.
* Because it is a normalised average of $(\mathbf{y}_j - \mathbf{x})$, the mean-shift vector always points *into*
  the convex hull of the samples. It cannot send you off to infinity.

In [ ]:
def mean_shift(x, samples, tau, kernel="lap"):
    """Normalised mean-shift vector at each row of x.
    x: [N, D], samples: [M, D] -> [N, D]"""
    if kernel == "lap":
        logits = -torch.cdist(x, samples) / tau
    else:
        logits = -torch.cdist(x, samples) ** 2 / (2 * tau ** 2)
    w = logits.softmax(-1)                 # [N, M], the normalised kernel k-bar
    return w @ samples - x                 # sum_j kbar_j y_j  -  x

# visualise the mean-shift field of the bimodal data
gq = torch.linspace(-4.5, 4.5, 21)
QX, QY = torch.meshgrid(gq, gq, indexing="xy")
Q = torch.stack([QX.reshape(-1), QY.reshape(-1)], -1)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, tau in zip(ax, [0.3, 0.8, 2.5]):
    V = mean_shift(Q, data, tau)
    a.contourf(GX, GY, kde(grid, data, 0.5).reshape(GX.shape), levels=25, cmap="Greys")
    a.quiver(Q[:, 0], Q[:, 1], V[:, 0], V[:, 1], color="crimson", width=0.004)
    a.set_title(f"mean-shift field, tau={tau}"); a.set_aspect("equal")
plt.tight_layout(); plt.show()

At small $\tau$ the field points to the nearest local cluster; at large $\tau$ everything is dragged to the
global centre of mass — *straight into the empty valley between the two modes*. Keep that failure mode in mind: it
is the "flat kernel" pathology the paper reports when it says drifting fails on ImageNet in raw pixel space.

Let's confirm that iterating mean shift is mode seeking.

In [ ]:
traj = [torch.tensor([[-0.2, 3.2], [3.5, -2.5], [0.0, 0.0]])]
for _ in range(40):
    traj.append(traj[-1] + mean_shift(traj[-1], data, tau=0.6))
traj = torch.stack(traj)                     # [T, 3, 2]

plt.figure(figsize=(5, 5))
plt.scatter(data[:, 0], data[:, 1], s=6, c="lightgray")
for i in range(traj.shape[1]):
    plt.plot(traj[:, i, 0], traj[:, i, 1], "-o", ms=2.5, lw=1)
    plt.scatter(*traj[0, i], marker="s", s=45, zorder=5)
    plt.scatter(*traj[-1, i], marker="*", s=160, zorder=5)
plt.title("iterated mean shift converges to modes\n(squares = start, stars = end)")
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.show()

## 4. Attraction and repulsion: the drifting field, ahead of schedule

Mean shift alone climbs to a mode and **stops there**. That is mode collapse — as a generative model it is useless.
The drifting insight is to build a *second* mean-shift field from your **own current samples** and subtract it:

$$\boxed{\;\mathbf{V}_{p,q}(\mathbf{x}) = \underbrace{\mathbf{V}^{+}_{p}(\mathbf{x})}_{\text{attraction to data}} - \underbrace{\mathbf{V}^{-}_{q}(\mathbf{x})}_{\text{repulsion from self}}\;}$$

When your samples $q$ look exactly like the data $p$, the two terms are computed from statistically identical
clouds and cancel: $\mathbf{V} = \mathbf{0}$. That is the **equilibrium**. Let's see it.

In [ ]:
def drift_field(x, y_pos, y_neg, tau, a_pos=1.0, a_neg=1.0, exclude_self=False):
    Vp = mean_shift(x, y_pos, tau)
    if exclude_self:
        d = torch.cdist(x, y_neg) + torch.eye(x.shape[0]) * 1e6
        Vn = (-d / tau).softmax(-1) @ y_neg - x
    else:
        Vn = mean_shift(x, y_neg, tau)
    return a_pos * Vp - a_neg * Vn

p_data = make_bimodal(300)
q_bad  = make_bimodal(300, sep=0.0, sd=0.6)          # collapsed in the middle
q_good = make_bimodal(300)                           # already matches p

fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))
for a, (q, name) in zip(ax, [(q_bad, "q collapsed in the middle"),
                             (make_bimodal(300, sep=3.5, sd=0.3), "q too spread out"),
                             (q_good, "q matches p  (equilibrium)")]):
    V = drift_field(Q, p_data, q, tau=0.8)
    a.scatter(p_data[:, 0], p_data[:, 1], s=7, c="tab:blue", alpha=0.5, label="p (data, +)")
    a.scatter(q[:, 0], q[:, 1], s=7, c="tab:orange", alpha=0.5, label="q (generated, -)")
    a.quiver(Q[:, 0], Q[:, 1], V[:, 0], V[:, 1], color="k", width=0.004,
             scale=None, angles="xy")
    a.set_title(f"{name}\nmean |V| = {V.norm(dim=-1).mean():.3f}")
    a.set_aspect("equal"); a.legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.show()

Read the three panels:

* **left** — $q$ sits in the valley; the field points *outward toward both modes*. Note it does not collapse onto one:
  the repulsion from its own crowded centre pushes samples apart while the two data modes pull them in opposite directions.
* **middle** — $q$ is over-dispersed; the field points *inward*.
* **right** — $q\approx p$; the field is tiny and directionless. Only sampling noise remains.

The mean $\|\mathbf{V}\|$ printed in each title is a stand-in for the drifting **loss** (which is the mean of $\|\mathbf{V}\|^2$). That is the entire
training signal — a scalar that measures "how far is my sample cloud from equilibrium".

## 5. Flowing points along a vector field

A vector field $\mathbf{V}$ plus a step size $\eta$ defines a discrete flow $\mathbf{x} \leftarrow \mathbf{x} + \eta\,\mathbf{V}(\mathbf{x})$.
Iterate it on a *whole cloud at once*, recomputing $\mathbf{V}$ from the current cloud each step, and you have an
**interacting particle system**. This is what drifting does — the only twist (Notebook 1) is that the iteration index
is the *training step*, not an inference step.

In [ ]:
def flow(q0, p, steps=120, eta=0.6, tau=0.6):
    q = q0.clone(); hist = [q.clone()]
    for _ in range(steps):
        V = drift_field(q, p, q, tau=tau, exclude_self=True)
        q = q + eta * V
        hist.append(q.clone())
    return torch.stack(hist)

q0 = 0.25 * torch.randn(300, 2)                    # start collapsed at the origin
H = flow(q0, p_data)
drift_norm = [drift_field(h, p_data, h, tau=0.6, exclude_self=True).pow(2).sum(-1).mean().item()
              for h in H]

fig = plt.figure(figsize=(14, 3.4))
for i, t in enumerate([0, 5, 20, 60, 120]):
    a = fig.add_subplot(1, 6, i + 1)
    a.scatter(p_data[:, 0], p_data[:, 1], s=4, c="tab:blue", alpha=0.35)
    a.scatter(H[t][:, 0], H[t][:, 1], s=4, c="tab:orange")
    a.set_title(f"step {t}", fontsize=9); a.set_xlim(-4.5, 4.5); a.set_ylim(-3, 3)
    a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
a = fig.add_subplot(1, 6, 6)
a.semilogy(drift_norm); a.set_title("mean ||V||^2", fontsize=9); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

A cloud initialised as a single collapsed blob at the origin recovers **both** modes, and the mean squared drift
decays by orders of magnitude. No neural network was involved. Everything a drifting model does is contained in
this picture; the network's only job (Notebook 1) is to *memorise the endpoint of this flow as a function of the
input noise*, so that a single forward pass reproduces it.

## 6. Conservative vs non-conservative fields (and why the kernel choice matters)

A vector field is **conservative** if it is the gradient of a scalar potential, $\mathbf{V} = \nabla \phi$.
Conservative fields have zero curl; non-conservative fields can circulate forever.

There is a beautiful identity here. For the **Gaussian** kernel,
$\nabla_{\mathbf{x}} K_h(\mathbf{x}-\mathbf{y}) = \frac{\mathbf{y}-\mathbf{x}}{h^2} K_h(\mathbf{x}-\mathbf{y})$, so

$$\nabla_{\mathbf{x}}\log \rho_h(\mathbf{x}) = \frac{\sum_j (\mathbf{y}_j-\mathbf{x}) K_h}{h^2\sum_j K_h} = \frac{1}{h^2}\,\mathbf{V}^{+}(\mathbf{x})$$

**The Gaussian mean-shift vector is exactly $h^2$ times the score of the KDE.** So the Gaussian drifting field is
$h^2\big(\nabla\log\rho_{p,h} - \nabla\log\rho_{q,h}\big)$ — a gradient of $\log(\rho_p/\rho_q)$, hence conservative.
For the Laplace kernel the identity fails (an extra $1/\|\mathbf{y}-\mathbf{x}\|$ appears) and the field genuinely circulates.

Let's verify both claims numerically.

In [ ]:
# --- claim 1: Gaussian mean shift == tau^2 * score of the KDE ---
def kde_logsum(z, samples, tau):
    d2 = ((z[:, None, :] - samples[None, :, :]) ** 2).sum(-1)
    return torch.logsumexp(-d2 / (2 * tau ** 2), dim=1)     # constants drop out of the gradient

Zq = torch.randn(8, 2, requires_grad=True)
tau = 0.6
score = torch.autograd.grad(kde_logsum(Zq, p_data, tau).sum(), Zq)[0]
ms = mean_shift(Zq.detach(), p_data, tau, kernel="gauss")
print("max |mean_shift - tau^2 * score| =", (ms - tau ** 2 * score).abs().max().item())

In [ ]:
# --- claim 2: curl of the drifting field, Gaussian vs Laplace ---
Ypos = 0.5 * torch.randn(150, 2) + torch.tensor([1.5, 0.0])
Yneg = 0.5 * torch.randn(150, 2) + torch.tensor([-1.5, 0.5])

def V_of(kind):
    def f(z, tau=0.7):
        return (mean_shift(z, Ypos, tau, kernel=kind) - mean_shift(z, Yneg, tau, kernel=kind))
    return f

def curl2d(field, pts, eps=1e-3):
    ex, ey = torch.tensor([eps, 0.0]), torch.tensor([0.0, eps])
    dVy_dx = (field(pts + ex)[:, 1] - field(pts - ex)[:, 1]) / (2 * eps)
    dVx_dy = (field(pts + ey)[:, 0] - field(pts - ey)[:, 0]) / (2 * eps)
    return dVy_dx - dVx_dy

probes = 1.2 * torch.randn(400, 2)
for kind in ["gauss", "lap"]:
    f = V_of(kind)
    c = curl2d(f, probes).abs().mean()
    s = f(probes).norm(dim=-1).mean()
    print(f"{kind:6s}: mean|curl| = {c:.3e}   mean|V| = {s:.3e}   relative = {c/s:.2e}")

The Gaussian field's curl is at the level of finite-difference noise ($\sim10^{-4}$ relative); the Laplace field's
is over a hundred times larger ($1.4\times10^{-2}$). The paper's field is **not** a gradient flow — it is a genuinely
non-conservative interacting-particle system. That is not a bug (empirically Laplace works better), but it *is* the
reason the convergence theory is harder; see `THEORY.md` §7 and the finite-particle analysis of
[arXiv:2605.22795](https://arxiv.org/abs/2605.22795).

### The same vector is Tweedie's denoiser

Diffusion models learn the same object from the other side. Tweedie's formula says that if $\mathbf{y}=\mathbf{x}+\sigma\boldsymbol{\xi}$ with $\mathbf{x}\sim p$ and $\boldsymbol{\xi}\sim\mathcal{N}(0,I)$, the best denoiser (the posterior mean) is

136653\mathbb{E}[\mathbf{x}\mid\mathbf{y}] = \mathbf{y}+\sigma^{2}\nabla_{\mathbf{y}}\log p_\sigma(\mathbf{y}),\qquad p_\sigma = \mathcal{N}(0,\sigma^2 I)*p.136653

When $ is an empirical set, \sigma$ is *exactly* the Gaussian KDE with bandwidth =\sigma$, so by the identity above
$\mathbb{E}[\mathbf{x}\mid\mathbf{y}]-\mathbf{y}=\mathbf{V}^{+}(\mathbf{y})$: **the attraction half of the drifting field is
the ideal denoiser's displacement at noise level $\tau$.** The check below does not use the identity: it draws \times10^{6}$
noisy samples, keeps those that land within /bin/bash.15$ of a query point, and averages their clean originals.

In [ ]:
# Tweedie: Monte-Carlo posterior mean E[x | y ~ y0] versus y0 + (Gaussian mean-shift vector at y0)
tau, M, rho = 0.6, 4_000_000, 0.15
y0 = torch.randn(4, 2)
x_clean = p_data[torch.randint(0, len(p_data), (M,))]
y_noisy = x_clean + tau * torch.randn(M, 2)
pred = y0 + mean_shift(y0, p_data, tau, kernel="gauss")
for q in range(4):
    m = (y_noisy - y0[q]).norm(dim=-1) < rho
    post, se = x_clean[m].mean(0), x_clean[m].std(0).max() / m.sum().sqrt()
    print(f"query {q}: n={int(m.sum()):6d}   max|posterior mean - (y0 + V+)| = {(post - pred[q]).abs().max():.4f}   MC std-err {se:.4f}")

The discrepancy is within two Monte-Carlo standard errors (a ball of radius $\rho$ also adds an (\rho^2)$ bias, small here). So one
attraction-only drifting step at $\tau=\sigma$ *is* one ideal denoising step toward the data. What drifting adds is the repulsion
$\mathbf{V}^{-}_q$, and the fact that the field is applied to the generator's own samples *during training*, not to a corrupted
data point at inference. `THEORY.md` §7.1 gives the identity; Notebook 12 uses the same field as a per-sample score.

## 7. Why softmax normalisation is not optional

The mean-shift vector divides by $Z(\mathbf{x})=\sum_j k(\mathbf{x},\mathbf{y}_j)$. Drop that and you get the
**un-normalised** field, which is what an MMD gradient gives you (Notebook 6). The difference is dramatic when
$\mathbf{x}$ is far from everything: the normalised field still points somewhere sensible, the un-normalised field
goes to zero and the sample is stranded.

In [ ]:
xs = torch.stack([torch.linspace(0, 12, 60), torch.zeros(60)], -1)   # walk away from the data
w_unnorm = torch.exp(-torch.cdist(xs, p_data) / 0.8)
V_norm   = mean_shift(xs, p_data, tau=0.8)
V_unnorm = (w_unnorm[..., None] * (p_data[None] - xs[:, None])).sum(1) / p_data.shape[0]

plt.figure(figsize=(6, 3.4))
plt.semilogy(xs[:, 0], V_norm.norm(dim=-1) + 1e-30, label="normalised (mean shift)")
plt.semilogy(xs[:, 0], V_unnorm.norm(dim=-1) + 1e-30, label="un-normalised (MMD-like)")
plt.xlabel("distance of x from the data"); plt.ylabel("|V|"); plt.legend(); plt.grid(alpha=0.3)
plt.title("normalisation keeps the field alive far from the data"); plt.show()

The normalised field's magnitude *grows* with distance (it always points at the data). The un-normalised field
dies exponentially. This is one of the four concrete advantages over MMD that the paper lists, and it is why the
authors report being unable to make an MMD formulation work at all.

## What we have built

| concept | formula | code |
|---|---|---|
| Laplace kernel | $\exp(-\|x-y\|/\tau)$ | `k_lap` |
| normalised kernel | $\mathrm{softmax}_j(-d_j/\tau)$ | `logits.softmax(-1)` |
| mean-shift vector | $\sum_j\bar k_j(y_j-x)$ | `mean_shift` |
| drifting field | $V^{+}_{p}-V^{-}_{q}$ | `drift_field` |
| particle flow | $x \leftarrow x+\eta V(x)$ | `flow` |
| equilibrium | $V=0 \iff$ (roughly) $q=p$ | the right-hand panel of §4 |

Next: [0b. Pushforward Maps and Distances Between Distributions](<0b. Pushforward Maps and Distances Between Distributions.ipynb>).